# Historical results and comparison

**Historical experimental results from the original project experiments.** No corrected-pipeline benchmarks have been run.

Classical DSP is the pilot-only reference. DeepRx represents convolutional reception; DAT represents a more compute-intensive attention baseline. The hybrid combines physical correction with lightweight attention.

The CSVs preserve printed precision, source notebook/cell (zero-based), and Git commit. Blank fields mean unreported values. Baseline implementations have been removed; their observations remain here.

## Protocols and limitations

- `original_seed`: 2,000 frames/SNR, data and noise seed 46, also used in training. Post-FEC metrics at 0–12 dB in 1 dB steps; pre-FEC BER only in 2 dB steps.
- `fresh_seed`: 1,024 frames/SNR, data seed 999 and noise seed 888, at 0–12 dB in 2 dB steps. Only post-FEC BER was printed.
- Historical neural decoding used `10**(Eb/N0 / 10)/4` above 6 dB and `1 + Eb/N0*0.5` otherwise. Current evaluation uses one fixed multiplier, default 1.0.
- The hybrid measurements predate the corrected CFO/time reference and independent training streams. The fresh-seed comparison is not a rerun of those corrections.
- Classical curves are taken consistently from notebook 09. DAT's separately printed classical values differ slightly; they are not averaged or substituted.
- Zero recorded errors are finite-sample observations, not zero error probability.

Excluded: smart routing/classical-estimator injection (including 2.734e−4 at 12 dB), unrelated architectures/ablations, and conflicting summary-plot claims. No values were extrapolated.

In [ ]:
import csv
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "results").is_dir() else Path.cwd().parent
with (ROOT / "results/benchmark_results.csv").open() as f:
    benchmark = list(csv.DictReader(f))
with (ROOT / "results/complexity_results.csv").open() as f:
    complexity = list(csv.DictReader(f))
models = ["Classical DSP", "DeepRx (CNN)", "DAT (attention)", "Hybrid LA (ours)"]

## Selected observations

Original-seed protocol:

| Method | Post-FEC BER @ 6 dB | Post-FEC BER @ 12 dB | BLER @ 12 dB |
| --- | --- | --- | --- |
| Classical DSP | 1.594e-01 | 8.593e-02 | 4.020e-01 |
| DeepRx (CNN) | 1.477e-01 | 1.004e-02 | 7.400e-02 |
| DAT (attention) | 3.037e-03 | 0.000e+00 | 0.000e+00 |
| Hybrid LA (ours) | 3.765e-02 | 5.513e-03 | 1.050e-02 |

Fresh-seed protocol:

| Method | Post-FEC BER @ 6 dB | Post-FEC BER @ 12 dB |
| --- | --- | --- |
| Classical DSP | 1.631e-01 | 8.704e-02 |
| DeepRx (CNN) | 1.925e-01 | 3.396e-02 |
| DAT (attention) | 4.243e-03 | 0.000e+00 |
| Hybrid LA (ours) | 8.393e-02 | 3.025e-02 |

These static tables were populated from the CSVs for reading without execution.

In [ ]:
# Display the same selected values directly from the CSVs.
for protocol in ("original_seed", "fresh_seed"):
    lines = [f"### {protocol}", "| Method | Eb/N0 | Post-FEC BER | BLER |",
             "| --- | --- | --- | --- |"]
    for row in benchmark:
        if row["protocol"] == protocol and row["ebn0_db"] in ("6", "12"):
            lines.append("| " + " | ".join(row[k] or "—" for k in
                         ("model", "ebn0_db", "ber_post_info", "bler_post_info")) + " |")
    display(Markdown("\n".join(lines)))

In [ ]:
# Linear y-axes retain recorded zeros without inventing a logarithmic floor.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, metric in zip(axes, ("ber_pre_coded", "ber_post_info", "bler_post_info")):
    for model in models:
        rows = sorted((r for r in benchmark if r["protocol"] == "original_seed"
                       and r["model"] == model and r[metric]), key=lambda r: float(r["ebn0_db"]))
        ax.plot([float(r["ebn0_db"]) for r in rows],
                [float(r[metric]) for r in rows], ".-", label=model)
    ax.set(xlabel="Eb/N0 (dB)", ylabel=metric, title="Historical: original seed")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for model in models:
    rows = [r for r in benchmark if r["protocol"] == "fresh_seed" and r["model"] == model]
    ax.plot([float(r["ebn0_db"]) for r in rows],
            [float(r["ber_post_info"]) for r in rows], ".-", label=model)
ax.set(xlabel="Eb/N0 (dB)", ylabel="Post-FEC BER", title="Historical: fresh seeds")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Computational cost

| Method | Parameters | Approx. MFLOPs | ms/sample |
| --- | ---: | ---: | ---: |
| Hybrid LA (ours) | 276,888 | 72.57 | 0.08764 |
| DeepRx (CNN) | 297,924 | 343.72 | 0.12247 |
| DAT (attention) | 289,652 | 147.81 | 0.45562 |

Approximate FLOPs are the original THOP report (twice counted MACs); unsupported operations may be omitted. They are not full receiver FLOPs. Latency is CUDA neural forward time/sample at batch size 128, aggregated over the fresh-seed sweep with the first two batches discarded. It excludes generation and LDPC. GPU models were not recorded in these outputs, so do not interpret timings as a controlled hardware ranking. Classical cost was not reported.

In [ ]:
lines = ["| Method | Parameters | MFLOPs | ms/sample ± std |",
         "| --- | --- | --- | --- |"]
for r in complexity:
    lines.append(f"| {r['model']} | {r['parameters']} | {r['flops_million']} | "
                 f"{r['latency_ms_per_sample']} ± {r['latency_std_ms']} |")
display(Markdown("\n".join(lines)))

## Interpretation

The hybrid's historical results illustrate a useful performance/compute trade-off relative to the CNN baseline. DAT achieved better decoding results with greater reported compute. The gap between original-seed and fresh-seed hybrid results also matters: it limits conclusions about generalization.

The next valid performance claim must come from rerunning the corrected pipeline. This notebook only presents existing observations.